# SHAP — Explicabilidad del mejor modelo

En este notebook aplicamos SHAP al mejor modelo identificado en el notebook 8
(**Exp 0: LogisticRegression + TF-IDF**, F1-macro test: 0.9530) para entender
qué palabras/features impulsan cada clase de riesgo del AI Act.

Pasos:
1. Carga del mejor modelo y datos
2. Cálculo de valores SHAP
3. Beeswarm plot por clase (top 20 features)
4. Bar plot global de importancia media
5. Waterfall plot de ejemplos individuales
6. Interpretación
7. Registro en MLflow

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Carga del mejor modelo y datos

In [3]:
import pandas as pd
import joblib
from functions import crear_tfidf

# Datos
train_df = pd.read_csv("data/processed/train.csv")
test_df  = pd.read_csv("data/processed/test.csv")

X_train = train_df["text_final"]
y_train = train_df["etiqueta"]
X_test  = test_df["text_final"]
y_test  = test_df["etiqueta"]

# Mejor modelo (Exp 0: LogReg + TF-IDF baseline)
modelo = joblib.load("model/modelo_baseline.joblib")
tfidf  = joblib.load("model/tfidf_vectorizer.joblib")

# Vectorizar
X_train_tfidf = tfidf.transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

feature_names = tfidf.get_feature_names_out().tolist()
class_names   = sorted(modelo.classes_.tolist())

print(f"Train: {X_train_tfidf.shape}")
print(f"Test:  {X_test_tfidf.shape}")
print(f"Features: {len(feature_names)}")
print(f"Clases: {class_names}")

Train: (210, 3811)
Test:  (45, 3811)
Features: 3811
Clases: ['alto_riesgo', 'inaceptable', 'riesgo_limitado', 'riesgo_minimo']


## 2. Cálculo de valores SHAP

In [5]:
from functions import explicar_con_shap

# X_train_tfidf como referencia (background), X_test_tfidf como muestras a explicar
explainer, shap_values = explicar_con_shap(
    modelo,
    X_background=X_train_tfidf,
    X_explain=X_test_tfidf,
)

print(f"Tipo de explainer: {type(explainer).__name__}")
print(f"Número de clases: {len(shap_values)}")
print(f"Shape por clase: {shap_values[0].shape}")

ModuleNotFoundError: No module named 'shap'

## 3. Beeswarm plot por clase (top 20 features)

Cada punto es una muestra del test set. El color indica el valor de la feature
(rojo = alto, azul = bajo). La posición horizontal indica el impacto en la predicción.

In [ ]:
from functions import plot_shap_summary

saved_paths = plot_shap_summary(
    shap_values=shap_values,
    X_explain=X_test_tfidf,
    feature_names=feature_names,
    class_names=class_names,
    output_dir="model",
    max_display=20,
)

print(f"\nPlots guardados: {saved_paths}")

## 4. Waterfall plot de ejemplos individuales

Elegimos una muestra de cada clase del test set para visualizar
cómo contribuye cada feature a esa predicción concreta.

In [ ]:
import numpy as np
from functions import plot_shap_waterfall

# Seleccionar un índice representativo por clase
indices_por_clase = {}
for clase in class_names:
    mask = y_test.values == clase
    if mask.any():
        indices_por_clase[clase] = np.where(mask)[0][0]

print("Índices seleccionados por clase:")
for clase, idx in indices_por_clase.items():
    pred = modelo.predict(X_test_tfidf[idx])[0]
    print(f"  {clase}: idx={idx} | predicho={pred} | texto={X_test.iloc[idx][:80]}...")

In [ ]:
waterfall_paths = []

for clase, idx in indices_por_clase.items():
    print(f"\n--- Waterfall para clase '{clase}' (índice {idx}) ---")
    paths = plot_shap_waterfall(
        explainer=explainer,
        shap_values=shap_values,
        X_explain=X_test_tfidf,
        idx=idx,
        feature_names=feature_names,
        class_names=class_names,
        output_dir="model",
        max_display=15,
    )
    waterfall_paths.extend(paths)

## 5. Interpretación

Documentar aquí las observaciones tras revisar los plots:

- **inaceptable**: ¿qué palabras empujan hacia esta clase?
- **alto_riesgo**: ¿qué términos son más discriminativos?
- **riesgo_limitado**: ¿qué señales usa el modelo?
- **riesgo_minimo**: ¿qué vocabulario es característico?

Coherencia con el AI Act:
- ¿Los features SHAP coinciden con las palabras clave del dominio (`KEYWORDS_DOMINIO`)?
- ¿El modelo está aprendiendo reglas regulatorias o patrones espurios?

## 6. Registro en MLflow

In [ ]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
from functions import log_mlflow_safe

all_shap_artifacts = saved_paths + waterfall_paths

try:
    log_mlflow_safe(
        run_name="shap_explicabilidad",
        params={
            "modelo":         "LogisticRegression",
            "explainer_tipo": "LinearExplainer",
            "n_muestras":     X_test_tfidf.shape[0],
            "n_features":     len(feature_names),
            "max_display":    20,
        },
        tags={"experimento": "0", "fase": "explicabilidad"},
        artifacts=all_shap_artifacts,
    )
    print("✓ SHAP registrado en MLflow")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")